In [ ]:
!git clone https://github.com/va1shn9v/PromptIR.git
%cd PromptIR
!pip install einops pytorch_lightning==2.0.0 opencv-python tqdm

Cloning into 'PromptIR'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 126 (delta 45), reused 37 (delta 37), pack-reused 70 (from 1)
Receiving objects: 100% (126/126), 1.39 MiB | 4.78 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/PromptIR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 715.6/715.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 57.0 MB/s eta 0:00:00


In [ ]:
import urllib.request, os
os.makedirs("ckpt", exist_ok=True)
urllib.request.urlretrieve(
    "https://github.com/va1shn9v/PromptIR/releases/download/v1.0/model.ckpt",
    "ckpt/model.ckpt"
)
print("Downloaded!")

Downloaded!


In [ ]:
import os, time
import cv2, numpy as np, torch
import torch.nn.functional as F
from tqdm import tqdm
import pytorch_lightning as pl
from net.model import PromptIR

class PromptIRModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = PromptIR(decoder=True)
    def forward(self, x):
        return self.net(x)

def load_model(ckpt_path, device):
    print(f"[INFO] Loading checkpoint: {ckpt_path}")
    model = PromptIRModel()
    checkpoint = torch.load(ckpt_path, map_location=device)
    if "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    else:
        model.load_state_dict(checkpoint)
    model.to(device)
    model.eval()
    print("[INFO] Model loaded.")
    return model

def pad_to_multiple(tensor, multiple=8):
    _, h, w = tensor.shape
    pad_h = (multiple - h % multiple) % multiple
    pad_w = (multiple - w % multiple) % multiple
    if pad_h > 0 or pad_w > 0:
        tensor = F.pad(tensor, (0, pad_w, 0, pad_h), mode="reflect")
    return tensor, (pad_h, pad_w)

def gaussian_window(size):
    sigma = size / 6.0
    coords = torch.arange(size, dtype=torch.float32) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g2d = g.unsqueeze(0) * g.unsqueeze(1)
    return g2d / g2d.max()

def infer_tiled(model, img_tensor, tile, overlap, device):
    c, h, w = img_tensor.shape
    output = torch.zeros_like(img_tensor)
    weight = torch.zeros(1, h, w, dtype=torch.float32)
    step = tile - overlap
    ys = sorted(set(max(0, y) for y in list(range(0, h - tile + 1, step)) + [h - tile]))
    xs = sorted(set(max(0, x) for x in list(range(0, w - tile + 1, step)) + [w - tile]))
    if h <= tile and w <= tile:
        patch, _ = pad_to_multiple(img_tensor, 8)
        with torch.no_grad():
            out = model(patch.unsqueeze(0).to(device)).squeeze(0).cpu()
        return out[:, :h, :w].clamp(0, 1)
    blend = gaussian_window(tile)
    for y in ys:
        y2 = min(y + tile, h)
        for x in xs:
            x2 = min(x + tile, w)
            patch = img_tensor[:, y:y2, x:x2]
            patch_padded, _ = pad_to_multiple(patch, 8)
            with torch.no_grad():
                restored = model(patch_padded.unsqueeze(0).to(device)).squeeze(0).cpu()
            restored = restored[:, :y2-y, :x2-x]
            bh, bw = y2-y, x2-x
            w_patch = blend[:bh, :bw]
            output[:, y:y2, x:x2] += restored * w_patch
            weight[:, y:y2, x:x2] += w_patch
    return (output / weight.clamp(min=1e-6)).clamp(0, 1)

print("All functions defined!")

All functions defined!


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = load_model("ckpt/model.ckpt", device)

Using device: cuda
[INFO] Loading checkpoint: ckpt/model.ckpt
[INFO] Model loaded.


In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
img_name = list(uploaded.keys())[0]

img = Image.open(img_name).convert('RGB')
img.save("input_image.jpg")
print(f"Image ready: {img.size}")

Saving rain-over-forest-mountains-misty-260nw-2271121319.webp to rain-over-forest-mountains-misty-260nw-2271121319.webp
Image ready: (390, 280)


In [ ]:
from torchvision import transforms

img = Image.open("input_image.jpg").convert('RGB')
h, w = img.size[1], img.size[0]
print(f"Image size: {w}x{h}")

img_tensor = transforms.ToTensor()(img)

# Run with tiling (handles any resolution)
restored = infer_tiled(model, img_tensor, tile=512, overlap=32, device=device)

# Save raw PromptIR output
result = transforms.ToPILImage()(restored.clamp(0, 1))
result.save("restored_image.png")
print("PromptIR done!")

# Enhancement
frame = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)
frame = cv2.convertScaleAbs(frame, alpha=1.3, beta=10)
lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)
clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8,8))
l = clahe.apply(l)
frame = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
kernel = np.array([[ 0, -0.5,  0], [-0.5, 3, -0.5], [ 0, -0.5,  0]])
frame = cv2.filter2D(frame, -1, kernel)
enhanced = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
enhanced.save("enhanced_image.png")
print("Enhancement done!")

Image size: 390x280
PromptIR done!
Enhancement done!


In [ ]:

files.download("enhanced_image.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>